# FinRL PPO Portfolio Demo v2

Demo: user portfolio input, predefined strategy, FinRL data/env, custom high-level actions, PPO, and TradingView-like plots.

Raw FinRL `StockTradingEnv` uses continuous per-stock actions. This notebook wraps it with high-level demo actions: `HOLD`, `BUY`, `SELL`, `REBALANCE`, `CHANGE_STRATEGY`.

In [ ]:
# Install if needed
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git
# !pip install stable-baselines3 gymnasium shimmy yfinance pandas numpy plotly



In [ ]:
!pip install plotly

In [1]:
import os
import warnings


import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import gymnasium as gym
    from gymnasium import spaces
except Exception:
    import gym
    from gym import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.config import INDICATORS
from finrl.plot import backtest_stats

c:\Users\gurug\miniconda3\envs\stockdss\Lib\site-packages\pyfolio\pos.py:25: UserWarning: Module "zipline.assets" not found; multipliers will not be applied to position notionals.
  warnings.warn(


In [2]:
""" # ------------------------------------------------------------
# USER INPUT
# ------------------------------------------------------------

portfolio_input = {
    "AAPL": 10,
    "MSFT": 5,
    "NVDA": 3,
}

SELECTED_STRATEGY_ID = "balanced"  # defensive | balanced | aggressive
ALLOW_CHANGE_STRATEGY_ACTION = True

TRAIN_START_DATE = "2020-01-01"
TRAIN_END_DATE   = "2023-12-31"

TRADE_START_DATE = "2024-01-01"
TRADE_END_DATE   = "2025-12-31"

INITIAL_CASH = 100_000
FALLBACK_CSV = "stockdata.csv"

USE_VIX = False
USE_TURBULENCE = False
TOTAL_TIMESTEPS = 20_000

CHART_TICKER = "AAPL"

tickers = list(portfolio_input.keys())
initial_num_stock_shares = [portfolio_input[tic] for tic in tickers]

print("Tickers:", tickers)
print("Initial shares:", initial_num_stock_shares)
print("Strategy:", SELECTED_STRATEGY_ID) """

' # ------------------------------------------------------------\n# USER INPUT\n# ------------------------------------------------------------\n\nportfolio_input = {\n    "AAPL": 10,\n    "MSFT": 5,\n    "NVDA": 3,\n}\n\nSELECTED_STRATEGY_ID = "balanced"  # defensive | balanced | aggressive\nALLOW_CHANGE_STRATEGY_ACTION = True\n\nTRAIN_START_DATE = "2020-01-01"\nTRAIN_END_DATE   = "2023-12-31"\n\nTRADE_START_DATE = "2024-01-01"\nTRADE_END_DATE   = "2025-12-31"\n\nINITIAL_CASH = 100_000\nFALLBACK_CSV = "stockdata.csv"\n\nUSE_VIX = False\nUSE_TURBULENCE = False\nTOTAL_TIMESTEPS = 20_000\n\nCHART_TICKER = "AAPL"\n\ntickers = list(portfolio_input.keys())\ninitial_num_stock_shares = [portfolio_input[tic] for tic in tickers]\n\nprint("Tickers:", tickers)\nprint("Initial shares:", initial_num_stock_shares)\nprint("Strategy:", SELECTED_STRATEGY_ID) '

In [ ]:
# ------------------------------------------------------------
# INTERACTIVE USER INPUT
# ------------------------------------------------------------

# If needed:
# !pip install ipywidgets

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date

portfolio_rows = []
portfolio_box = widgets.VBox()
output = widgets.Output()

# Strategy options must match keys in STRATEGIES cell below
strategy_dropdown = widgets.Dropdown(
    options=[
        ("Defensive", "defensive"),
        ("Balanced", "balanced"),
        ("Aggressive", "aggressive"),
    ],
    value="balanced",
    description="Strategy:",
)

risk_slider = widgets.FloatSlider(
    value=0.50,
    min=0.00,
    max=1.00,
    step=0.01,
    description="Risk:",
    readout=True,
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="400px")
)

allow_change_strategy_checkbox = widgets.Checkbox(
    value=True,
    description="Allow change_strategy"
)

train_start_picker = widgets.DatePicker(
    description="Train start:",
    value=date(2020, 1, 1)
)

train_end_picker = widgets.DatePicker(
    description="Train end:",
    value=date(2023, 12, 31)
)

trade_start_picker = widgets.DatePicker(
    description="Trade start:",
    value=date(2024, 1, 1)
)

trade_end_picker = widgets.DatePicker(
    description="Trade end:",
    value=date(2025, 12, 31)
)

initial_cash_input = widgets.IntText(
    value=100_000,
    description="Cash:"
)

fallback_csv_input = widgets.Text(
    value="stockdata.csv",
    description="CSV:"
)

use_vix_checkbox = widgets.Checkbox(
    value=False,
    description="Use VIX"
)

use_turbulence_checkbox = widgets.Checkbox(
    value=False,
    description="Use Turbulence"
)

timesteps_input = widgets.IntText(
    value=20_000,
    description="Timesteps:"
)

chart_ticker_dropdown = widgets.Dropdown(
    options=["AAPL", "MSFT", "NVDA"],
    value="AAPL",
    description="Chart:"
)


def refresh_portfolio_box():
    portfolio_box.children = [row["widget"] for row in portfolio_rows]

    current_tickers = [
        row["ticker"].value.upper().strip()
        for row in portfolio_rows
        if row["ticker"].value.strip()
    ]

    if current_tickers:
        chart_ticker_dropdown.options = current_tickers

        if chart_ticker_dropdown.value not in current_tickers:
            chart_ticker_dropdown.value = current_tickers[0]
    else:
        chart_ticker_dropdown.options = []


def add_portfolio_row(ticker_value="", shares_value=0):
    ticker_input = widgets.Text(
        value=ticker_value,
        placeholder="AAPL",
        description="Ticker:"
    )

    shares_input = widgets.IntText(
        value=shares_value,
        description="Shares:"
    )

    remove_button = widgets.Button(
        description="-",
        button_style="danger",
        layout=widgets.Layout(width="40px")
    )

    row_widget = widgets.HBox([
        ticker_input,
        shares_input,
        remove_button
    ])

    row = {
        "ticker": ticker_input,
        "shares": shares_input,
        "widget": row_widget
    }

    def remove_row(_):
        portfolio_rows.remove(row)
        refresh_portfolio_box()

    remove_button.on_click(remove_row)

    portfolio_rows.append(row)
    refresh_portfolio_box()


add_button = widgets.Button(
    description="+ Add stock",
    button_style="success"
)

save_button = widgets.Button(
    description="Save input config",
    button_style="primary"
)


def on_add_clicked(_):
    add_portfolio_row()


def on_save_clicked(_):
    global portfolio_input
    global SELECTED_STRATEGY_ID
    global RISK_LEVEL
    global ALLOW_CHANGE_STRATEGY_ACTION

    global TRAIN_START_DATE
    global TRAIN_END_DATE
    global TRADE_START_DATE
    global TRADE_END_DATE

    global INITIAL_CASH
    global FALLBACK_CSV
    global USE_VIX
    global USE_TURBULENCE
    global TOTAL_TIMESTEPS
    global CHART_TICKER

    global tickers
    global initial_num_stock_shares

    portfolio_input = {}

    for row in portfolio_rows:
        ticker = row["ticker"].value.upper().strip()
        shares = int(row["shares"].value)

        if ticker and shares >= 0:
            portfolio_input[ticker] = shares

    SELECTED_STRATEGY_ID = strategy_dropdown.value
    RISK_LEVEL = float(risk_slider.value)
    ALLOW_CHANGE_STRATEGY_ACTION = allow_change_strategy_checkbox.value

    TRAIN_START_DATE = train_start_picker.value.strftime("%Y-%m-%d")
    TRAIN_END_DATE = train_end_picker.value.strftime("%Y-%m-%d")

    TRADE_START_DATE = trade_start_picker.value.strftime("%Y-%m-%d")
    TRADE_END_DATE = trade_end_picker.value.strftime("%Y-%m-%d")

    INITIAL_CASH = int(initial_cash_input.value)
    FALLBACK_CSV = fallback_csv_input.value

    USE_VIX = bool(use_vix_checkbox.value)
    USE_TURBULENCE = bool(use_turbulence_checkbox.value)
    TOTAL_TIMESTEPS = int(timesteps_input.value)

    CHART_TICKER = chart_ticker_dropdown.value

    tickers = list(portfolio_input.keys())
    initial_num_stock_shares = [portfolio_input[tic] for tic in tickers]

    with output:
        clear_output()

        print("Saved config")
        print("------------")
        print("portfolio_input =", portfolio_input)
        print("SELECTED_STRATEGY_ID =", SELECTED_STRATEGY_ID)
        print("RISK_LEVEL =", RISK_LEVEL)
        print("ALLOW_CHANGE_STRATEGY_ACTION =", ALLOW_CHANGE_STRATEGY_ACTION)
        print("TRAIN_START_DATE =", TRAIN_START_DATE)
        print("TRAIN_END_DATE =", TRAIN_END_DATE)
        print("TRADE_START_DATE =", TRADE_START_DATE)
        print("TRADE_END_DATE =", TRADE_END_DATE)
        print("INITIAL_CASH =", INITIAL_CASH)
        print("FALLBACK_CSV =", FALLBACK_CSV)
        print("USE_VIX =", USE_VIX)
        print("USE_TURBULENCE =", USE_TURBULENCE)
        print("TOTAL_TIMESTEPS =", TOTAL_TIMESTEPS)
        print("CHART_TICKER =", CHART_TICKER)
        print("tickers =", tickers)
        print("initial_num_stock_shares =", initial_num_stock_shares)


add_button.on_click(on_add_clicked)
save_button.on_click(on_save_clicked)

# Default portfolio
add_portfolio_row("AAPL", 10)
add_portfolio_row("MSFT", 5)
add_portfolio_row("NVDA", 3)

display(
    widgets.VBox([
        widgets.HTML("<h3>Portfolio</h3>"),
        portfolio_box,
        add_button,

        widgets.HTML("<h3>Strategy</h3>"),
        strategy_dropdown,
        risk_slider,
        allow_change_strategy_checkbox,

        widgets.HTML("<h3>Train period</h3>"),
        train_start_picker,
        train_end_picker,

        widgets.HTML("<h3>Trade/Test period</h3>"),
        trade_start_picker,
        trade_end_picker,

        widgets.HTML("<h3>Other settings</h3>"),
        initial_cash_input,
        fallback_csv_input,
        use_vix_checkbox,
        use_turbulence_checkbox,
        timesteps_input,
        chart_ticker_dropdown,

        save_button,
        output
    ])
)

In [4]:
STRATEGIES = {
    "defensive": {
        "strategy_id": "defensive",
        "display_name": "Defensive",
        "risk_profile": "low",
        "objective": "capital_preservation",
        "description": "Capital preservation, lower drawdown, lower concentration, higher cash buffer.",
        "constraints": {
            "allow_shorting": False,
            "allow_margin": False,
            "max_position_weight": 0.15,
            "min_cash_weight": 0.15,
            "target_cash_weight": 0.2,
            "max_turnover_per_decision": 0.2
        },
        "risk_policy": {
            "lambda_drawdown": 1.25,
            "lambda_volatility": 1.0,
            "lambda_transaction_cost": 0.5,
            "lambda_concentration": 1.25,
            "lambda_strategy_violation": 2.0,
            "score_quantile": "q25"
        },
        "allowed_actions": [
            "HOLD",
            "REBALANCE",
            "SELL",
            "BUY",
            "CHANGE_STRATEGY"
        ],
        "rebalance": {
            "frequency": "monthly",
            "drift_threshold": 0.05
        }
    },
    "balanced": {
        "strategy_id": "balanced",
        "display_name": "Balanced",
        "risk_profile": "medium",
        "objective": "risk_adjusted_growth",
        "description": "Balanced portfolio growth with downside-risk control, moderate cash, and controlled concentration.",
        "constraints": {
            "allow_shorting": False,
            "allow_margin": False,
            "max_position_weight": 0.25,
            "min_cash_weight": 0.05,
            "target_cash_weight": 0.1,
            "max_turnover_per_decision": 0.35
        },
        "risk_policy": {
            "lambda_drawdown": 0.75,
            "lambda_volatility": 0.5,
            "lambda_transaction_cost": 0.35,
            "lambda_concentration": 0.75,
            "lambda_strategy_violation": 1.5,
            "score_quantile": "q50"
        },
        "allowed_actions": [
            "HOLD",
            "REBALANCE",
            "SELL",
            "BUY",
            "CHANGE_STRATEGY"
        ],
        "rebalance": {
            "frequency": "monthly",
            "drift_threshold": 0.08
        }
    },
    "aggressive": {
        "strategy_id": "aggressive",
        "display_name": "Aggressive",
        "risk_profile": "high",
        "objective": "upside_growth",
        "description": "Upside and growth while accepting higher volatility, drawdown risk, and concentration.",
        "constraints": {
            "allow_shorting": False,
            "allow_margin": False,
            "max_position_weight": 0.35,
            "min_cash_weight": 0.0,
            "target_cash_weight": 0.03,
            "max_turnover_per_decision": 0.5
        },
        "risk_policy": {
            "lambda_drawdown": 0.35,
            "lambda_volatility": 0.25,
            "lambda_transaction_cost": 0.25,
            "lambda_concentration": 0.3,
            "lambda_strategy_violation": 1.0,
            "score_quantile": "q75"
        },
        "allowed_actions": [
            "HOLD",
            "REBALANCE",
            "SELL",
            "BUY",
            "CHANGE_STRATEGY"
        ],
        "rebalance": {
            "frequency": "monthly",
            "drift_threshold": 0.12
        }
    }
}

selected_strategy = STRATEGIES[SELECTED_STRATEGY_ID]
pd.DataFrame([selected_strategy['constraints'], selected_strategy['risk_policy']], index=['constraints', 'risk_policy']).T

,constraints,risk_policy
allow_shorting,False,NaN
allow_margin,False,NaN
max_position_weight,0.35,NaN
min_cash_weight,0.0,NaN
target_cash_weight,0.03,NaN
max_turnover_per_decision,0.5,NaN
lambda_drawdown,NaN,0.35
lambda_volatility,NaN,0.25
lambda_transaction_cost,NaN,0.25
lambda_concentration,NaN,0.3


In [5]:
def load_market_data(
    tickers,
    start_date,
    end_date,
    fallback_csv="stockdata.csv",
    download=True
):
    if download:
        try:
            print("Trying FinRL YahooDownloader...")
            df = YahooDownloader(
                start_date=start_date,
                end_date=end_date,
                ticker_list=tickers
            ).fetch_data()
            print("Downloaded data from Yahoo via FinRL.")
            return df

        except Exception as e:
            print("Yahoo/FinRL download failed:", e)
            print(f"Using fallback CSV: {fallback_csv}")

    else:
        print(f"DOWNLOAD_DATA=False, using fallback CSV directly: {fallback_csv}")

    if not os.path.exists(fallback_csv):
        raise FileNotFoundError(f"Fallback CSV not found: {fallback_csv}")

    df = pd.read_csv(fallback_csv)

    df = df[df["tic"].isin(tickers)].copy()
    df["date"] = pd.to_datetime(df["date"])

    df = df[
        (df["date"] >= pd.to_datetime(start_date)) &
        (df["date"] <= pd.to_datetime(end_date))
    ].copy()

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    return df

In [6]:
DOWNLOAD_DATA = False  # True = Yahoo/FinRL, False = direkte fallback CSV

In [7]:
raw_df = load_market_data(
    tickers=tickers,
    start_date=TRAIN_START_DATE,
    end_date=TRADE_END_DATE,
    fallback_csv=FALLBACK_CSV,
    download=DOWNLOAD_DATA
)

raw_df = raw_df.sort_values(["date", "tic"]).reset_index(drop=True)

print(raw_df.shape)

raw_df.groupby("tic")["date"].agg(["min", "max", "count"])

DOWNLOAD_DATA=False, using fallback CSV directly: stockdata.csv
(4524, 17)


,min,max,count
tic,,,
AAPL,2020-01-02,2025-12-31,1508
MSFT,2020-01-02,2025-12-31,1508
TSLA,2020-01-02,2025-12-31,1508


In [8]:
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=USE_VIX,
    use_turbulence=USE_TURBULENCE,
    user_defined_feature=False,
)

processed = fe.preprocess_data(raw_df)

# Sort by date and tic to ensure correct order for the environment
# Fix duplicate indicator columns created as _x / _y
for indicator in INDICATORS:
    x_col = f"{indicator}_x"
    y_col = f"{indicator}_y"

    if indicator not in processed.columns:
        if x_col in processed.columns:
            processed[indicator] = processed[x_col]
        elif y_col in processed.columns:
            processed[indicator] = processed[y_col]

# Drop duplicate suffixed columns
cols_to_drop = [
    col for col in processed.columns
    if col.endswith("_x") or col.endswith("_y")
]

processed = processed.drop(columns=cols_to_drop)

processed = processed.sort_values(["date", "tic"]).reset_index(drop=True)


processed = processed.sort_values(["date", "tic"]).reset_index(drop=True)
processed.head()

Successfully added technical indicators


,date,tic,close,high,low,open,volume,day,vix,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma
0,2020-01-02,AAPL,72.333878,72.394086,71.091184,71.344054,135480400.0,3.0,12.47,2.106708,72.255603,62.219656,76.464881,176.869284,58.873244,66.039411,62.756884
1,2020-01-02,MSFT,152.158386,152.262592,149.989032,150.415323,22622100.0,3.0,12.47,2.636741,152.936207,140.688204,69.649569,144.289972,42.731275,145.379384,139.955711
2,2020-01-02,TSLA,28.684000,28.713333,28.114000,28.299999,142981500.0,3.0,12.47,1.671410,30.545789,20.726078,72.769524,120.902519,40.967594,24.639889,22.293344
3,2020-01-03,AAPL,71.630646,72.389265,71.406674,71.563213,146322800.0,4.0,14.02,2.155225,72.642083,62.692707,73.464437,153.876311,58.873244,66.289395,63.052712
4,2020-01-03,MSFT,150.263748,151.523684,149.733252,149.979564,21116200.0,4.0,14.02,2.555215,153.085901,141.369307,65.103422,111.932203,40.078659,145.655370,140.325325


In [9]:
train = data_split(processed, TRAIN_START_DATE, TRAIN_END_DATE)
trade = data_split(processed, TRADE_START_DATE, TRADE_END_DATE)

print("Train:", train["date"].min(), train["date"].max(), train.shape)
print("Trade:", trade["date"].min(), trade["date"].max(), trade.shape)

Train: 2020-01-02 2023-12-29 (3018, 17)
Trade: 2024-01-02 2025-12-30 (1503, 17)


In [10]:
stock_dimension = len(tickers)
state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension

strategy_constraints = selected_strategy["constraints"]
strategy_hmax = max(1, int(100 * strategy_constraints["max_turnover_per_decision"]))

env_kwargs = {
    "hmax": strategy_hmax,
    "initial_amount": INITIAL_CASH,
    "num_stock_shares": initial_num_stock_shares,
    "buy_cost_pct": [0.001] * stock_dimension,
    "sell_cost_pct": [0.001] * stock_dimension,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
}

env_kwargs

{'hmax': 50,
 'initial_amount': 100000,
 'num_stock_shares': [250, 250, 250],
 'buy_cost_pct': [0.001, 0.001, 0.001],
 'sell_cost_pct': [0.001, 0.001, 0.001],
 'state_space': 31,
 'stock_dim': 3,
 'tech_indicator_list': ['macd',
  'boll_ub',
  'boll_lb',
  'rsi_30',
  'cci_30',
  'dx_30',
  'close_30_sma',
  'close_60_sma'],
 'action_space': 3,
 'reward_scaling': 0.0001}

In [11]:
class StrategyActionWrapper(gym.Env):
    ACTION_NAMES = ["HOLD", "BUY", "SELL", "REBALANCE", "CHANGE_STRATEGY"]

    def __init__(self, base_env, tickers, strategies, selected_strategy_id="balanced", allow_change_strategy=True):
        super().__init__()
        self.base_env = base_env
        self.tickers = list(tickers)
        self.stock_dim = len(tickers)
        self.strategies = strategies
        self.strategy_ids = list(strategies.keys())
        self.current_strategy_id = selected_strategy_id
        self.allow_change_strategy = allow_change_strategy
        self.action_space = spaces.Discrete(len(self.ACTION_NAMES))
        self.observation_space = getattr(base_env, "observation_space", spaces.Box(low=-np.inf, high=np.inf, shape=(base_env.state_space,), dtype=np.float32))
        self.action_log = []

    @property
    def current_strategy(self):
        return self.strategies[self.current_strategy_id]

    def reset(self, seed=None, options=None):
        try:
            result = self.base_env.reset(seed=seed)
        except TypeError:
            result = self.base_env.reset()
        self.action_log = []
        if isinstance(result, tuple):
            obs, info = result
        else:
            obs, info = result, {}
        return np.asarray(obs, dtype=np.float32), info

    def step(self, action):
        action = int(action)
        action_name = self.ACTION_NAMES[action]
        vector_action = self._map_action(action_name)
        result = self.base_env.step(vector_action)

        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
        else:
            obs, reward, done, info = result
            terminated, truncated = done, False

        penalty = self._strategy_penalty()
        reward = float(reward) + penalty
        row = {
            "date": self._current_date(),
            "action_id": action,
            "action_name": action_name,
            "strategy_id": self.current_strategy_id,
            "mapped_finrl_action": vector_action.tolist(),
            "strategy_penalty": penalty,
            "reward": reward,
            "account_value": self._account_value(),
        }
        self.action_log.append(row)
        info = dict(info or {})
        info.update(row)
        return np.asarray(obs, dtype=np.float32), reward, terminated, truncated, info

    def _map_action(self, action_name):
        turnover = float(self.current_strategy["constraints"]["max_turnover_per_decision"])
        intensity = np.clip(turnover, 0.05, 1.0)
        if action_name == "HOLD":
            return np.zeros(self.stock_dim, dtype=np.float32)
        if action_name == "BUY":
            return np.ones(self.stock_dim, dtype=np.float32) * intensity
        if action_name == "SELL":
            return -np.ones(self.stock_dim, dtype=np.float32) * intensity
        if action_name == "REBALANCE":
            return self._rebalance_action_vector()
        if action_name == "CHANGE_STRATEGY":
            if self.allow_change_strategy:
                i = self.strategy_ids.index(self.current_strategy_id)
                self.current_strategy_id = self.strategy_ids[(i + 1) % len(self.strategy_ids)]
            return np.zeros(self.stock_dim, dtype=np.float32)
        raise ValueError(action_name)

    def _rebalance_action_vector(self):
        weights = self._position_weights()
        target_cash = float(self.current_strategy["constraints"].get("target_cash_weight", 0.1))
        target_stock_weight = (1.0 - target_cash) / self.stock_dim
        diff = target_stock_weight - weights
        return np.clip(diff * 5.0, -1.0, 1.0).astype(np.float32)

    def _strategy_penalty(self):
        weights = self._position_weights()
        max_w = float(self.current_strategy["constraints"].get("max_position_weight", 1.0))
        lambda_violation = float(self.current_strategy["risk_policy"].get("lambda_strategy_violation", 1.0))
        violation = np.maximum(weights - max_w, 0).sum()
        return -lambda_violation * float(violation) * 0.01

    def _position_weights(self):
        state = np.asarray(getattr(self.base_env, "state", []), dtype=float)
        if len(state) < 1 + 2 * self.stock_dim:
            return np.ones(self.stock_dim) / self.stock_dim
        prices = state[1:1 + self.stock_dim]
        shares = state[1 + self.stock_dim:1 + 2 * self.stock_dim]
        values = np.maximum(prices, 0) * np.maximum(shares, 0)
        total = values.sum() + max(state[0], 0)
        if total <= 0:
            return np.zeros(self.stock_dim)
        return values / total

    def _account_value(self):
        if hasattr(self.base_env, "asset_memory") and len(self.base_env.asset_memory) > 0:
            return float(self.base_env.asset_memory[-1])
        state = np.asarray(getattr(self.base_env, "state", []), dtype=float)
        if len(state) >= 1 + 2 * self.stock_dim:
            prices = state[1:1 + self.stock_dim]
            shares = state[1 + self.stock_dim:1 + 2 * self.stock_dim]
            return float(state[0] + np.sum(prices * shares))
        return np.nan

    def _current_date(self):
        if hasattr(self.base_env, "date_memory") and len(self.base_env.date_memory) > 0:
            return self.base_env.date_memory[-1]
        if hasattr(self.base_env, "df") and hasattr(self.base_env, "day"):
            dates = sorted(pd.Series(self.base_env.df["date"]).unique())
            day = min(int(self.base_env.day), len(dates) - 1)
            return dates[day]
        return None

In [12]:
print(INDICATORS)
print(processed.columns.tolist())

missing = [col for col in INDICATORS if col not in processed.columns]
print("Missing indicators:", missing)

['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']
['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'day', 'vix', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']
Missing indicators: []


In [13]:
def make_train_env():
    base_env = StockTradingEnv(df=train, **env_kwargs)
    return StrategyActionWrapper(base_env, tickers, STRATEGIES, SELECTED_STRATEGY_ID, ALLOW_CHANGE_STRATEGY_ACTION)

env_train = DummyVecEnv([make_train_env])

model_ppo = PPO(
    policy="MlpPolicy",
    env=env_train,
    n_steps=2048,
    batch_size=128,
    ent_coef=0.01,
    learning_rate=0.00025,
    verbose=1,
)

model_ppo.learn(total_timesteps=TOTAL_TIMESTEPS)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1355 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1227        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010586565 |
|    clip_fraction        | 0.0487      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | 0.0134      |
|    learning_rate        | 0.00025     |
|    loss                 | 2.55        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00561    |
|    value_loss           | 5.58        |
-----------------------------------------
-----------------

In [14]:
def run_strategy_backtest(model):
    base_trade_env = StockTradingEnv(df=trade, **env_kwargs)
    wrapped_trade_env = StrategyActionWrapper(base_trade_env, tickers, STRATEGIES, SELECTED_STRATEGY_ID, ALLOW_CHANGE_STRATEGY_ACTION)
    obs, info = wrapped_trade_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = wrapped_trade_env.step(int(action))
        done = terminated or truncated
    action_log = pd.DataFrame(wrapped_trade_env.action_log)
    if hasattr(base_trade_env, "save_asset_memory"):
        df_account_value = base_trade_env.save_asset_memory()
    else:
        df_account_value = action_log[["date", "account_value"]].copy()
    return df_account_value, action_log


df_account_value, df_actions = run_strategy_backtest(model_ppo)
display(df_account_value.tail())
display(df_actions.tail())

,date,account_value
496,2025-12-23,475641.210013
497,2025-12-24,476569.462943
498,2025-12-26,472150.552177
499,2025-12-29,465773.898521
500,2025-12-30,463512.485068


,date,action_id,action_name,strategy_id,mapped_finrl_action,strategy_penalty,reward,account_value
496,2025-12-24,1,BUY,aggressive,"[0.5, 0.5, 0.5]",-0.000660,0.092165,476569.462943
497,2025-12-26,1,BUY,aggressive,"[0.5, 0.5, 0.5]",-0.000642,-0.442533,472150.552177
498,2025-12-29,1,BUY,aggressive,"[0.5, 0.5, 0.5]",-0.000606,-0.638272,465773.898521
499,2025-12-30,1,BUY,aggressive,"[0.5, 0.5, 0.5]",-0.000601,-0.226742,463512.485068
500,2025-12-30,1,BUY,aggressive,"[0.5, 0.5, 0.5]",-0.000601,-0.226742,463512.485068


In [15]:
try:
    perf_stats = backtest_stats(account_value=df_account_value)
    #display(perf_stats)
except Exception as e:
    print("backtest_stats failed:", e)
    display(df_account_value.tail())

Annual return          0.246410
Cumulative returns     0.549468
Annual volatility      0.292139
Sharpe ratio           0.900422
Calmar ratio           0.665142
Stability              0.755794
Max drawdown          -0.370462
Omega ratio            1.171040
Sortino ratio          1.353260
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.990184
Daily value at risk   -0.035762
dtype: float64


In [16]:
# ------------------------------------------------------------
# ZERO-SUM / TRADINGVIEW-LIKE CHART - STABLE IPYWIDGET TOOLBAR
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, clear_output
import traceback
import ipywidgets as widgets


# ------------------------------------------------------------
# Shared styling
# ------------------------------------------------------------

PAPER_BG = "#f3eee5"
PLOT_BG = "#f3eee5"
GRID = "#d7d0c3"
TEXT = "#262626"
MUTED = "#7a766e"
BORDER = "#1f1f1f"
GREEN = "#2f7d3a"
RED = "#c92a35"
GOLD = "#a87928"
BLUE = "#2d6fb7"
GREY = "#6f6f6a"

PLOTLY_CONFIG = {
    "displayModeBar": "hover",
    "displaylogo": False,
    "modeBarButtonsToRemove": [
        "lasso2d",
        "select2d",
        "autoScale2d",
        "toggleSpikelines",
    ],
}

# Backtest "now": by default the simulated current time is the end of the trade/test period.
if "NOW_DATE" not in globals():
    NOW_DATE = globals().get("TRADE_END_DATE", None)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _strategy_text(selected_strategy=None, risk_level=None):
    text = "PPO strategy backtest"

    if selected_strategy is not None:
        text = f"Strategy: {selected_strategy.get('display_name', '')}"

    if risk_level is not None:
        text += f" · Risk: {risk_level:.2f}"

    return text


def _date_range(df, period):
    max_date = pd.to_datetime(df["date"]).max()

    if period == "1M":
        return max_date - pd.DateOffset(months=1), max_date
    if period == "6M":
        return max_date - pd.DateOffset(months=6), max_date
    if period == "YTD":
        return pd.Timestamp(year=max_date.year, month=1, day=1), max_date
    if period == "1Y":
        return max_date - pd.DateOffset(years=1), max_date
    if period == "2Y":
        return max_date - pd.DateOffset(years=2), max_date

    return None


def _apply_shared_layout(fig, height=850, top_margin=55):
    fig.update_layout(
        title=None,
        template="plotly_white",
        height=height,
        paper_bgcolor=PAPER_BG,
        plot_bgcolor=PLOT_BG,
        font=dict(
            color=TEXT,
            family="Courier New, monospace",
            size=12,
        ),
        margin=dict(l=60, r=70, t=top_margin, b=45),
        hovermode="x unified",
        xaxis_rangeslider_visible=False,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(243,238,229,0.85)",
            bordercolor=GRID,
            borderwidth=1,
            font=dict(size=11, color=TEXT),
        ),
    )

    for axis_name in fig.layout:
        if axis_name.startswith("xaxis"):
            fig.layout[axis_name].update(
                showgrid=True,
                gridcolor=GRID,
                gridwidth=0.7,
                linecolor=BORDER,
                linewidth=1,
                tickfont=dict(color=MUTED, size=11),
                rangeslider_visible=False,
            )
        if axis_name.startswith("yaxis"):
            fig.layout[axis_name].update(
                showgrid=True,
                gridcolor=GRID,
                gridwidth=0.7,
                zeroline=False,
                linecolor=BORDER,
                linewidth=1,
                tickfont=dict(color=MUTED, size=11),
                side="right",
            )

    return fig


def _add_newspaper_borders(fig):
    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=0,
        x1=1,
        y0=1.0,
        y1=1.0,
        line=dict(color=BORDER, width=2),
    )

    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=0,
        x1=1,
        y0=-0.02,
        y1=-0.02,
        line=dict(color=BORDER, width=1),
    )

    return fig


# ------------------------------------------------------------
# KPI and period helpers
# ------------------------------------------------------------

def _fmt_money(value):
    try:
        value = float(value)
    except Exception:
        return "n/a"
    if abs(value) >= 1_000_000:
        return f"${value/1_000_000:,.2f}M"
    if abs(value) >= 1_000:
        return f"${value/1_000:,.1f}k"
    return f"${value:,.2f}"


def _fmt_pct(value, signed=True):
    try:
        value = float(value)
    except Exception:
        return "n/a"
    sign = "+" if signed and value > 0 else ""
    return f"{sign}{value:.2f}%"


def _safe_sharpe_from_values(values, periods_per_year=252):
    values = pd.Series(values).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    returns = values.pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    if len(returns) < 2:
        return np.nan
    std = returns.std()
    if std == 0 or pd.isna(std):
        return np.nan
    return float(np.sqrt(periods_per_year) * returns.mean() / std)


def _portfolio_kpis(account_df):
    acc = account_df.copy()
    acc["date"] = pd.to_datetime(acc["date"])
    acc = acc.sort_values("date")
    if acc.empty:
        return {}

    start_value = float(acc["account_value"].iloc[0])
    end_value = float(acc["account_value"].iloc[-1])
    period_return = ((end_value / start_value) - 1.0) * 100 if start_value else np.nan
    drawdown = (acc["account_value"] / acc["account_value"].cummax() - 1.0) * 100
    max_drawdown = float(drawdown.min()) if len(drawdown) else np.nan
    sharpe = _safe_sharpe_from_values(acc["account_value"])
    volatility = acc["account_value"].pct_change().std() * np.sqrt(252) * 100

    return {
        "start_value": start_value,
        "end_value": end_value,
        "period_return": period_return,
        "max_drawdown": max_drawdown,
        "sharpe": sharpe,
        "volatility": float(volatility) if not pd.isna(volatility) else np.nan,
        "start_date": acc["date"].iloc[0],
        "end_date": acc["date"].iloc[-1],
    }


def _ticker_kpis(df):
    if df.empty:
        return {}
    start_close = float(df["close"].iloc[0])
    last_close = float(df["close"].iloc[-1])
    period_return = ((last_close / start_close) - 1.0) * 100 if start_close else np.nan
    high = float(df["high"].max()) if "high" in df.columns else float(df["close"].max())
    low = float(df["low"].min()) if "low" in df.columns else float(df["close"].min())
    return {
        "start_close": start_close,
        "last_close": last_close,
        "period_return": period_return,
        "high": high,
        "low": low,
        "start_date": df["date"].iloc[0],
        "end_date": df["date"].iloc[-1],
    }


def _period_dates_from_globals():
    def _maybe_date(name):
        value = globals().get(name, None)
        if value is None or value == "":
            return None
        try:
            return pd.to_datetime(value)
        except Exception:
            return None

    train_start = _maybe_date("TRAIN_START_DATE")
    train_end = _maybe_date("TRAIN_END_DATE")
    trade_start = _maybe_date("TRADE_START_DATE")
    trade_end = _maybe_date("TRADE_END_DATE")
    now_date = _maybe_date("NOW_DATE") or trade_end

    return train_start, train_end, trade_start, trade_end, now_date


def _add_period_markers(fig, row="all", col="all"):
    """Add train/trade/now markers without failing if a chart has unusual axes."""
    train_start, train_end, trade_start, trade_end, now_date = _period_dates_from_globals()

    # Background zones. These make the backtest split visible.
    try:
        if train_start is not None and train_end is not None:
            fig.add_vrect(
                x0=train_start,
                x1=train_end,
                fillcolor="rgba(111,111,106,0.055)",
                line_width=0,
                layer="below",
                row=row,
                col=col,
            )
        if trade_start is not None and trade_end is not None:
            fig.add_vrect(
                x0=trade_start,
                x1=trade_end,
                fillcolor="rgba(168,121,40,0.055)",
                line_width=0,
                layer="below",
                row=row,
                col=col,
            )
    except Exception:
        pass

    markers = [
        (train_start, "TRAIN START", GREY, "dot"),
        (train_end, "TRAIN END", GREY, "dash"),
        (trade_start, "TRADE START", GOLD, "dash"),
        (trade_end, "TRADE END", GOLD, "dash"),
        (now_date, "NOW", RED, "solid"),
    ]

    seen = set()
    for x, label, color, dash in markers:
        if x is None:
            continue
        key = str(pd.to_datetime(x).date()) + label
        if key in seen:
            continue
        seen.add(key)
        try:
            fig.add_vline(
                x=x,
                line=dict(color=color, width=1.0, dash=dash),
                opacity=0.72,
                row=row,
                col=col,
            )
        except Exception:
            pass

    # Put compact labels above the first x-axis only.
    y = 1.015
    for x, label, color, dash in markers:
        if x is None:
            continue
        try:
            fig.add_annotation(
                x=x,
                y=y,
                xref="x",
                yref="paper",
                text=label,
                showarrow=False,
                font=dict(family="Courier New, monospace", size=9, color=color),
                bgcolor="rgba(243,238,229,0.85)",
                bordercolor=color,
                borderwidth=0.5,
                xanchor="center",
                yanchor="bottom",
            )
        except Exception:
            pass
    return fig


def _add_kpi_cards(fig, cards, y=1.125):
    """Add compact Zero-Sum style KPI cards in the top margin."""
    if not cards:
        return fig
    n = len(cards)
    gap = 0.010
    w = (1.0 - gap * (n - 1)) / n
    for i, (label, value, sub, tone) in enumerate(cards):
        x0 = i * (w + gap)
        x1 = x0 + w
        xm = (x0 + x1) / 2
        border = GREEN if tone == "positive" else RED if tone == "negative" else GRID
        value_color = GREEN if tone == "positive" else RED if tone == "negative" else TEXT
        fig.add_shape(
            type="rect",
            xref="paper",
            yref="paper",
            x0=x0,
            x1=x1,
            y0=y - 0.075,
            y1=y + 0.018,
            line=dict(color=border, width=0.8),
            fillcolor="rgba(243,238,229,0.92)",
            layer="above",
        )
        fig.add_annotation(
            x=xm,
            y=y - 0.012,
            xref="paper",
            yref="paper",
            text=(
                f"<span style='font-size:9px;color:{MUTED};letter-spacing:0.08em'>{label}</span>"
                f"<br><span style='font-size:15px;color:{value_color};font-weight:bold'>{value}</span>"
                f"<br><span style='font-size:9px;color:{MUTED}'>{sub}</span>"
            ),
            showarrow=False,
            align="center",
            xanchor="center",
            yanchor="middle",
        )
    return fig


# ------------------------------------------------------------
# Figure builders
# ------------------------------------------------------------

def build_single_ticker_chart(
    price_df,
    account_df=None,
    action_df=None,
    ticker="AAPL",
    selected_strategy=None,
    risk_level=None,
    period="ALL",
    chart_type="Candles",
    overlay_mode="MA 50/100/200",
    study_mode="Volume + Actions",
):
    df = price_df[price_df["tic"] == ticker].copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    if df.empty:
        raise ValueError(f"No data found for ticker: {ticker}")

    selected_range = _date_range(df, period)
    if selected_range is not None:
        start, end = selected_range
        df_view = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    else:
        df_view = df.copy()
    if df_view.empty:
        df_view = df.copy()

    # Indicators / overlays are calculated on full ticker history, then displayed in the selected period.
    df["ma50"] = df["close"].rolling(50).mean()
    df["ma100"] = df["close"].rolling(100).mean()
    df["ma200"] = df["close"].rolling(200).mean()
    df["ema8"] = df["close"].ewm(span=8, adjust=False).mean()
    df["ema21"] = df["close"].ewm(span=21, adjust=False).mean()
    rolling_mid = df["close"].rolling(20).mean()
    rolling_std = df["close"].rolling(20).std()
    df["bb_upper"] = rolling_mid + 2 * rolling_std
    df["bb_lower"] = rolling_mid - 2 * rolling_std

    if selected_range is not None:
        dfp = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    else:
        dfp = df.copy()
    if dfp.empty:
        dfp = df.copy()

    show_volume = study_mode in ["Volume + Actions", "Volume only"]
    show_actions = study_mode in ["Volume + Actions", "Actions only"]

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.035,
        row_heights=[0.64, 0.17, 0.19],
    )

    if chart_type in ["Candles", "Candles + Line"]:
        fig.add_trace(
            go.Candlestick(
                x=dfp["date"],
                open=dfp["open"],
                high=dfp["high"],
                low=dfp["low"],
                close=dfp["close"],
                name=ticker,
                increasing_line_color=GREEN,
                decreasing_line_color=RED,
                increasing_fillcolor="rgba(47, 125, 58, 0.50)",
                decreasing_fillcolor="rgba(201, 42, 53, 0.50)",
                line=dict(width=0.8),
            ),
            row=1,
            col=1,
        )

    if chart_type in ["Line", "Candles + Line"]:
        fig.add_trace(
            go.Scatter(
                x=dfp["date"],
                y=dfp["close"],
                mode="lines",
                name=f"{ticker} close",
                line=dict(color="#1f1f1f", width=1.25 if chart_type == "Line" else 0.85),
                opacity=1.0 if chart_type == "Line" else 0.72,
                hoverinfo="skip" if chart_type == "Candles + Line" else None,
            ),
            row=1,
            col=1,
        )

    if overlay_mode in ["MA 50/100/200", "All overlays"]:
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["ma50"], mode="lines", name="MA50", line=dict(color="#6f7377", width=1)), row=1, col=1)
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["ma100"], mode="lines", name="MA100", line=dict(color="#9b7fb0", width=1)), row=1, col=1)
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["ma200"], mode="lines", name="MA200", line=dict(color="#7fa6a4", width=1)), row=1, col=1)

    if overlay_mode in ["EMA 8/21", "All overlays"]:
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["ema8"], mode="lines", name="EMA8", line=dict(color="#b7791f", width=0.9)), row=1, col=1)
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["ema21"], mode="lines", name="EMA21", line=dict(color="#8b5a2b", width=0.9)), row=1, col=1)

    if overlay_mode in ["Bollinger", "All overlays"]:
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["bb_upper"], mode="lines", name="BB upper", line=dict(color="#7d8790", width=0.9, dash="dot")), row=1, col=1)
        fig.add_trace(go.Scatter(x=dfp["date"], y=dfp["bb_lower"], mode="lines", name="BB lower", line=dict(color="#7d8790", width=0.9, dash="dot")), row=1, col=1)

    last_close = float(dfp["close"].iloc[-1])
    fig.add_hline(
        y=last_close,
        line_dash="dot",
        line_color=RED if dfp["close"].iloc[-1] < dfp["open"].iloc[-1] else GREEN,
        line_width=1,
        row=1,
        col=1,
    )

    if show_actions and action_df is not None and len(action_df) > 0:
        actions = action_df.copy()
        actions["date"] = pd.to_datetime(actions["date"])
        if selected_range is not None:
            actions = actions[(actions["date"] >= start) & (actions["date"] <= end)].copy()
        merged = actions.merge(dfp[["date", "close"]], on="date", how="left").dropna(subset=["close"])

        marker_map = {
            "BUY": ("triangle-up", GREEN, "BUY"),
            "SELL": ("triangle-down", RED, "SELL"),
            "REBALANCE": ("diamond", BLUE, "REBALANCE"),
            "CHANGE_STRATEGY": ("star", GOLD, "CHANGE STRATEGY"),
            "HOLD": ("circle", GREY, "HOLD"),
        }

        for action_name, group in merged.groupby("action_name"):
            symbol, color, label = marker_map.get(action_name, ("circle", GREY, action_name))
            if action_name == "HOLD":
                group = group.tail(10)
            fig.add_trace(
                go.Scatter(
                    x=group["date"],
                    y=group["close"],
                    mode="markers",
                    name=label,
                    marker=dict(
                        symbol=symbol,
                        size=9 if action_name != "HOLD" else 6,
                        color=color,
                        opacity=0.85,
                        line=dict(width=1, color=BORDER),
                    ),
                    text=group.get("strategy_id", ""),
                    hovertemplate=(
                        "%{x|%Y-%m-%d}<br>"
                        f"Action: {label}<br>"
                        "Strategy: %{text}<br>"
                        "Price: %{y:.2f}"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )

    if show_volume:
        volume_colors = np.where(
            dfp["close"] >= dfp["open"],
            "rgba(47, 125, 58, 0.18)",
            "rgba(201, 42, 53, 0.18)",
        )
        fig.add_trace(
            go.Bar(
                x=dfp["date"],
                y=dfp["volume"],
                name="Volume",
                marker=dict(color=volume_colors, line=dict(width=0)),
                opacity=0.70,
                hovertemplate="%{x|%Y-%m-%d}<br>Volume: %{y:,}<extra></extra>",
            ),
            row=2,
            col=1,
        )

    if account_df is not None and len(account_df) > 0:
        acc = account_df.copy()
        acc["date"] = pd.to_datetime(acc["date"])
        acc = acc.sort_values("date")
        if selected_range is not None:
            acc = acc[(acc["date"] >= start) & (acc["date"] <= end)].copy()
        if len(acc) > 0:
            fig.add_trace(
                go.Scatter(
                    x=acc["date"],
                    y=acc["account_value"],
                    mode="lines",
                    name="Portfolio",
                    line=dict(color=GOLD, width=1.8),
                    hovertemplate="%{x|%Y-%m-%d}<br>Portfolio: %{y:,.2f}<extra></extra>",
                ),
                row=3,
                col=1,
            )
            min_acc = acc["account_value"].min()
            max_acc = acc["account_value"].max()
            padding = max((max_acc - min_acc) * 0.20, max_acc * 0.005)
            fig.update_yaxes(range=[min_acc - padding, max_acc + padding], row=3, col=1)

    kpis = _ticker_kpis(dfp)
    period_tone = "positive" if kpis.get("period_return", 0) >= 0 else "negative"
    cards = [
        ("NOW PRICE", _fmt_money(kpis.get("last_close", np.nan)), pd.to_datetime(kpis.get("end_date")).strftime("%Y-%m-%d") if kpis else "", "neutral"),
        ("PERIOD RETURN", _fmt_pct(kpis.get("period_return", np.nan)), f"from {_fmt_money(kpis.get('start_close', np.nan))}", period_tone),
        ("HIGH", _fmt_money(kpis.get("high", np.nan)), "selected period", "neutral"),
        ("LOW", _fmt_money(kpis.get("low", np.nan)), "selected period", "neutral"),
    ]

    fig.add_annotation(
        text=f"{ticker} — Price Chart",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.965,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )
    fig.add_annotation(
        text="Volume" if show_volume else "Volume hidden",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.310,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )
    fig.add_annotation(
        text="Portfolio Value",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.122,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )

    _apply_shared_layout(fig, height=900, top_margin=135)
    _add_kpi_cards(fig, cards, y=1.115)
    _add_period_markers(fig, row="all", col="all")
    _add_newspaper_borders(fig)

    return fig

def _latest_price_table(price_df):
    df = price_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["tic", "date"])
    return df.groupby("tic", as_index=False).tail(1).set_index("tic")


SECTOR_MAP = {
    # Technology / Communication-heavy demo map. Unknown tickers fall back to "Other".
    "AAPL": "Technology",
    "MSFT": "Technology",
    "NVDA": "Technology",
    "AMD": "Technology",
    "INTC": "Technology",
    "AVGO": "Technology",
    "GOOGL": "Communication Services",
    "GOOG": "Communication Services",
    "META": "Communication Services",
    "NFLX": "Communication Services",
    "AMZN": "Consumer Cyclical",
    "TSLA": "Consumer Cyclical",
    "JPM": "Financial Services",
    "BAC": "Financial Services",
    "V": "Financial Services",
    "MA": "Financial Services",
    "JNJ": "Healthcare",
    "UNH": "Healthcare",
    "ABBV": "Healthcare",
    "XOM": "Energy",
    "CVX": "Energy",
    "SPY": "Benchmark",
}


def _portfolio_holdings_summary(price_df, portfolio_input=None):
    if portfolio_input is None:
        if "portfolio_input" in globals():
            portfolio_input_local = globals()["portfolio_input"]
        else:
            raise ValueError("portfolio_input was not found.")
    else:
        portfolio_input_local = portfolio_input

    df = price_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["tic", "date"])

    rows = []
    for tic, shares in portfolio_input_local.items():
        tdf = df[df["tic"] == tic].copy()
        if tdf.empty:
            continue

        first_close = float(tdf["close"].iloc[0])
        last_close = float(tdf["close"].iloc[-1])
        shares = float(shares)
        initial_value = shares * first_close
        current_value = shares * last_close
        pnl = current_value - initial_value
        pct_return = ((last_close / first_close) - 1.0) * 100 if first_close else 0.0

        rows.append({
            "tic": tic,
            "shares": shares,
            "first_close": first_close,
            "last_close": last_close,
            "initial_value": initial_value,
            "current_value": current_value,
            "pnl": pnl,
            "pct_return": pct_return,
            "sector": SECTOR_MAP.get(tic, "Other"),
        })

    holdings = pd.DataFrame(rows)
    if holdings.empty:
        raise ValueError("No portfolio holdings could be matched against price_df.")

    total_value = holdings["current_value"].sum()
    holdings["weight_pct"] = np.where(total_value > 0, holdings["current_value"] / total_value * 100, 0.0)
    holdings = holdings.sort_values("current_value", ascending=False).reset_index(drop=True)
    return holdings


def _normalize_series(date_series, value_series):
    values = pd.Series(value_series).astype(float)
    first_valid = values.replace([np.inf, -np.inf], np.nan).dropna()
    if first_valid.empty or first_valid.iloc[0] == 0:
        return pd.Series(np.nan, index=values.index)
    return values / first_valid.iloc[0] * 100


def build_portfolio_value_chart(account_df, period="ALL"):
    """Backward-compatible portfolio chart: value + drawdown only."""
    if account_df is None or len(account_df) == 0:
        raise ValueError("No account value data found.")

    acc = account_df.copy()
    acc["date"] = pd.to_datetime(acc["date"])
    acc = acc.sort_values("date")
    acc["return_pct"] = acc["account_value"].pct_change().fillna(0)
    acc["drawdown_pct"] = (acc["account_value"] / acc["account_value"].cummax() - 1.0) * 100

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.70, 0.30],
    )

    fig.add_trace(
        go.Scatter(
            x=acc["date"],
            y=acc["account_value"],
            mode="lines",
            name="Portfolio value",
            line=dict(color=GOLD, width=2),
            hovertemplate="%{x|%Y-%m-%d}<br>Portfolio: %{y:,.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=acc["date"],
            y=acc["drawdown_pct"],
            mode="lines",
            name="Drawdown %",
            line=dict(color=RED, width=1.4),
            fill="tozeroy",
            fillcolor="rgba(201, 42, 53, 0.10)",
            hovertemplate="%{x|%Y-%m-%d}<br>Drawdown: %{y:.2f}%<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.add_annotation(
        text="Portfolio Value",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.965,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )

    fig.add_annotation(
        text="Drawdown",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.300,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )

    _apply_shared_layout(fig, height=760, top_margin=55)
    _add_newspaper_borders(fig)

    selected_range = _date_range(acc, period)
    if selected_range is not None:
        fig.update_xaxes(range=list(selected_range), row=1, col=1)
        fig.update_xaxes(range=list(selected_range), row=2, col=1)

    return fig


def build_portfolio_dashboard_chart(price_df, account_df, portfolio_input=None, period="ALL"):
    """Zero-Sum-inspired portfolio dashboard with KPI cards and train/trade/now markers."""
    if account_df is None or len(account_df) == 0:
        raise ValueError("No account value data found.")

    holdings = _portfolio_holdings_summary(price_df, portfolio_input=portfolio_input)

    acc = account_df.copy()
    acc["date"] = pd.to_datetime(acc["date"])
    acc = acc.sort_values("date")
    acc["drawdown_pct"] = (acc["account_value"] / acc["account_value"].cummax() - 1.0) * 100
    acc["portfolio_norm"] = _normalize_series(acc["date"], acc["account_value"])

    selected_range = _date_range(acc, period)
    if selected_range is not None:
        start, end = selected_range
        acc_view = acc[(acc["date"] >= start) & (acc["date"] <= end)].copy()
    else:
        acc_view = acc.copy()
    if acc_view.empty:
        acc_view = acc.copy()

    price = price_df.copy()
    price["date"] = pd.to_datetime(price["date"])
    price = price.sort_values(["tic", "date"])

    spy = price[price["tic"] == "SPY"].copy()
    has_spy = len(spy) > 0
    if has_spy:
        if selected_range is not None:
            spy = spy[(spy["date"] >= start) & (spy["date"] <= end)].copy()
        spy["spy_norm"] = _normalize_series(spy["date"], spy["close"])

    sector_alloc = (
        holdings.groupby("sector", as_index=False)["current_value"]
        .sum()
        .sort_values("current_value", ascending=False)
    )

    contributors = holdings.sort_values("pnl", ascending=True).copy()
    contributor_colors = np.where(contributors["pnl"] >= 0, GREEN, RED)

    kpis = _portfolio_kpis(acc_view)
    best_row = holdings.sort_values("pnl", ascending=False).iloc[0]
    worst_row = holdings.sort_values("pnl", ascending=True).iloc[0]
    return_tone = "positive" if kpis.get("period_return", 0) >= 0 else "negative"
    dd_tone = "negative" if kpis.get("max_drawdown", 0) < -5 else "neutral"

    sharpe_value = kpis.get("sharpe", np.nan)
    sharpe_text = "n/a" if pd.isna(sharpe_value) else f"{sharpe_value:.2f}"
    cards = [
        ("PORTFOLIO VALUE", _fmt_money(kpis.get("end_value", np.nan)), pd.to_datetime(kpis.get("end_date")).strftime("%Y-%m-%d"), "neutral"),
        ("PERIOD RETURN", _fmt_pct(kpis.get("period_return", np.nan)), f"from {_fmt_money(kpis.get('start_value', np.nan))}", return_tone),
        ("MAX DRAWDOWN", _fmt_pct(kpis.get("max_drawdown", np.nan), signed=False), "selected period", dd_tone),
        ("SHARPE RATIO", sharpe_text, f"vol {_fmt_pct(kpis.get('volatility', np.nan), signed=False)}", "neutral"),
        ("BEST CONTRIBUTOR", str(best_row["tic"]), _fmt_money(best_row["pnl"]), "positive" if best_row["pnl"] >= 0 else "negative"),
        ("WORST CONTRIBUTOR", str(worst_row["tic"]), _fmt_money(worst_row["pnl"]), "positive" if worst_row["pnl"] >= 0 else "negative"),
    ]

    fig = make_subplots(
        rows=4,
        cols=2,
        specs=[
            [{"type": "xy", "colspan": 2}, None],
            [{"type": "xy", "colspan": 2}, None],
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy"}, {"type": "xy"}],
        ],
        row_heights=[0.28, 0.17, 0.28, 0.27],
        vertical_spacing=0.075,
        horizontal_spacing=0.08,
        subplot_titles=(
            "Portfolio Value",
            "Drawdown",
            "Holdings Map / Allocation",
            "Sector Allocation",
            "Performance vs SPY",
            "Best / Worst Contributors",
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=acc_view["date"],
            y=acc_view["account_value"],
            mode="lines",
            name="Portfolio value",
            line=dict(color=GOLD, width=2),
            hovertemplate="%{x|%Y-%m-%d}<br>Portfolio: %{y:,.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=acc_view["date"],
            y=acc_view["drawdown_pct"],
            mode="lines",
            name="Drawdown %",
            line=dict(color=RED, width=1.2),
            fill="tozeroy",
            fillcolor="rgba(201, 42, 53, 0.10)",
            hovertemplate="%{x|%Y-%m-%d}<br>Drawdown: %{y:.2f}%<extra></extra>",
        ),
        row=2,
        col=1,
    )

    holdings_plot = holdings.sort_values("current_value", ascending=True).copy()
    holding_colors = np.where(holdings_plot["pct_return"] >= 0, GREEN, RED)
    fig.add_trace(
        go.Bar(
            x=holdings_plot["current_value"],
            y=holdings_plot["tic"],
            orientation="h",
            name="Holdings allocation",
            marker=dict(color=holding_colors, opacity=0.82, line=dict(color=PAPER_BG, width=1)),
            customdata=np.stack([holdings_plot["weight_pct"], holdings_plot["shares"], holdings_plot["pct_return"], holdings_plot["sector"]], axis=-1),
            text=holdings_plot["weight_pct"].map(lambda x: f"{x:.1f}%"),
            textposition="inside",
            insidetextanchor="middle",
            hovertemplate=(
                "<b>%{y}</b><br>Current value: $%{x:,.2f}<br>Weight: %{customdata[0]:.2f}%<br>Shares: %{customdata[1]:,.0f}<br>Return: %{customdata[2]:.2f}%<br>Sector: %{customdata[3]}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    sector_plot = sector_alloc.sort_values("current_value", ascending=True).copy()
    sector_total = float(sector_plot["current_value"].sum()) if len(sector_plot) else 0.0
    sector_plot["weight_pct"] = (sector_plot["current_value"] / sector_total) * 100.0 if sector_total > 0 else 0.0
    fig.add_trace(
        go.Bar(
            x=sector_plot["current_value"],
            y=sector_plot["sector"],
            orientation="h",
            name="Sector allocation",
            marker=dict(color=GREEN, opacity=0.80, line=dict(color=PAPER_BG, width=1)),
            customdata=np.stack([sector_plot["weight_pct"]], axis=-1),
            text=sector_plot["weight_pct"].map(lambda x: f"{x:.1f}%"),
            textposition="inside",
            insidetextanchor="middle",
            hovertemplate="<b>%{y}</b><br>Current value: $%{x:,.2f}<br>Weight: %{customdata[0]:.2f}%<extra></extra>",
        ),
        row=3,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=acc_view["date"],
            y=_normalize_series(acc_view["date"], acc_view["account_value"]),
            mode="lines",
            name="PPO Portfolio",
            line=dict(color=GOLD, width=2),
            hovertemplate="%{x|%Y-%m-%d}<br>Portfolio: %{y:.2f}<extra></extra>",
        ),
        row=4,
        col=1,
    )

    if has_spy and len(spy) > 0:
        fig.add_trace(
            go.Scatter(
                x=spy["date"],
                y=spy["spy_norm"],
                mode="lines",
                name="SPY",
                line=dict(color=GREY, width=1.5, dash="dot"),
                hovertemplate="%{x|%Y-%m-%d}<br>SPY: %{y:.2f}<extra></extra>",
            ),
            row=4,
            col=1,
        )
    else:
        fig.add_annotation(
            text="SPY not available in raw_df",
            xref="x5 domain",
            yref="y5 domain",
            x=0.5,
            y=0.5,
            showarrow=False,
            font=dict(family="Courier New, monospace", size=12, color=MUTED),
        )
    fig.add_hline(y=100, line_dash="dot", line_color=MUTED, line_width=1, row=4, col=1)

    fig.add_trace(
        go.Bar(
            x=contributors["pnl"],
            y=contributors["tic"],
            orientation="h",
            name="Contribution P/L",
            marker=dict(color=contributor_colors),
            hovertemplate="%{y}<br>P/L: $%{x:,.2f}<extra></extra>",
        ),
        row=4,
        col=2,
    )

    _apply_shared_layout(fig, height=1260, top_margin=160)
    _add_kpi_cards(fig, cards, y=1.110)
    _add_period_markers(fig, row="all", col="all")
    _add_newspaper_borders(fig)

    fig.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(243,238,229,0.85)",
            bordercolor=GRID,
            borderwidth=1,
            font=dict(size=11, color=TEXT),
        ),
    )

    for annotation in fig.layout.annotations:
        if isinstance(annotation.font, dict):
            pass
        annotation.font = annotation.font if annotation.font else dict(family="Georgia, serif", size=15, color=TEXT)

    fig.update_yaxes(ticksuffix="%", row=2, col=1)
    fig.update_xaxes(title_text="Current value", row=3, col=1)
    fig.update_yaxes(title_text="Ticker", row=3, col=1, autorange=True)
    fig.update_xaxes(title_text="Current value", row=3, col=2)
    fig.update_yaxes(title_text="Sector", row=3, col=2, autorange=True)
    fig.update_xaxes(title_text="Normalized performance", row=4, col=1)
    fig.update_yaxes(title_text="Rebased to 100", row=4, col=1)
    fig.update_xaxes(title_text="P/L contribution", row=4, col=2)

    return fig

def build_normalized_comparison_chart(price_df, account_df=None, selected_tickers=None, period="ALL"):
    df = price_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date", "tic"])

    if selected_tickers is None or len(selected_tickers) == 0:
        selected_tickers = sorted(df["tic"].unique())

    selected_range = _date_range(df, period)
    if selected_range is not None:
        start, end = selected_range
        df_view = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    else:
        df_view = df.copy()

    fig = go.Figure()

    for tic in selected_tickers:
        tdf = df_view[df_view["tic"] == tic].copy()
        if tdf.empty:
            continue

        base = float(tdf["close"].iloc[0])
        if base == 0:
            continue

        tdf["normalized"] = (tdf["close"] / base) * 100

        fig.add_trace(
            go.Scatter(
                x=tdf["date"],
                y=tdf["normalized"],
                mode="lines",
                name=tic,
                line=dict(width=1.4),
                hovertemplate="%{x|%Y-%m-%d}<br>%{fullData.name}: %{y:.2f}<extra></extra>",
            )
        )

    if account_df is not None and len(account_df) > 0:
        acc = account_df.copy()
        acc["date"] = pd.to_datetime(acc["date"])
        acc = acc.sort_values("date")

        if selected_range is not None:
            acc = acc[(acc["date"] >= start) & (acc["date"] <= end)].copy()

        if len(acc) > 0:
            base = float(acc["account_value"].iloc[0])
            if base != 0:
                acc["normalized"] = (acc["account_value"] / base) * 100

                fig.add_trace(
                    go.Scatter(
                        x=acc["date"],
                        y=acc["normalized"],
                        mode="lines",
                        name="PPO Portfolio",
                        line=dict(color=GOLD, width=2.2),
                        hovertemplate="%{x|%Y-%m-%d}<br>Portfolio: %{y:.2f}<extra></extra>",
                    )
                )

    fig.add_hline(y=100, line_dash="dot", line_color=MUTED, line_width=1)

    selected_label = ", ".join(selected_tickers)
    fig.add_annotation(
        text=f"Normalized comparison — rebased to 100 ({selected_label})",
        xref="paper",
        yref="paper",
        x=0.012,
        y=0.965,
        showarrow=False,
        font=dict(family="Georgia, serif", size=17, color=TEXT),
        xanchor="left",
    )

    # Period return summary for selected comparison window
    summary_parts = []
    for trace in fig.data:
        if hasattr(trace, "y") and len(trace.y) > 1:
            try:
                summary_parts.append(f"{trace.name}: {float(trace.y[-1]) - 100:+.1f}%")
            except Exception:
                pass
    if summary_parts:
        fig.add_annotation(
            text=" &nbsp; | &nbsp; ".join(summary_parts[:6]),
            xref="paper",
            yref="paper",
            x=0.012,
            y=1.075,
            showarrow=False,
            font=dict(family="Courier New, monospace", size=11, color=MUTED),
            xanchor="left",
        )

    _apply_shared_layout(fig, height=730, top_margin=95)
    _add_period_markers(fig)
    _add_newspaper_borders(fig)

    return fig


# ------------------------------------------------------------
# Stable external toolbar using ipywidgets
# ------------------------------------------------------------

available_tickers = list(dict.fromkeys(raw_df["tic"].tolist()))

if "tickers" in globals() and len(tickers) > 0:
    available_tickers = [tic for tic in tickers if tic in available_tickers]

if CHART_TICKER not in available_tickers and len(available_tickers) > 0:
    CHART_TICKER = available_tickers[0]

_default_compare_tickers = tuple(available_tickers[:min(3, len(available_tickers))])
if len(_default_compare_tickers) == 0:
    _default_compare_tickers = tuple(available_tickers)

header_html = widgets.HTML()

view_widget = widgets.ToggleButtons(
    options=["Single ticker", "Portfolio", "Comparison"],
    value="Single ticker",
    description="",
    button_style="",
)

# Used only in Single ticker view
ticker_widget = widgets.Dropdown(
    options=available_tickers,
    value=CHART_TICKER,
    description="",
    layout=widgets.Layout(width="260px"),
)
ticker_widget.add_class("zero-sum-ticker-dropdown")

# Used only in Comparison view
compare_tickers_widget = widgets.SelectMultiple(
    options=available_tickers,
    value=_default_compare_tickers,
    description="",
    layout=widgets.Layout(width="360px", height="92px"),
)
compare_tickers_widget.add_class("zero-sum-compare-select")

period_widget = widgets.ToggleButtons(
    options=["1M", "6M", "YTD", "1Y", "2Y", "ALL"],
    value="ALL",
    description="",
)

chart_widget = widgets.ToggleButtons(
    options=["Candles", "Line", "Candles + Line"],
    value="Candles",
    description="",
)

overlay_widget = widgets.ToggleButtons(
    options=["MA 50/100/200", "EMA 8/21", "Bollinger", "All overlays", "None"],
    value="MA 50/100/200",
    description="",
)

study_widget = widgets.ToggleButtons(
    options=["Volume + Actions", "Volume only", "Actions only", "Clean"],
    value="Volume + Actions",
    description="",
)

chart_output = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        min_width="100%",
        max_width="100%",
        overflow="visible",
    )
)

# Keep the whole widget toolbar area on the same Zero Sum background.
# ipywidgets Layout(background_color=...) is not reliable across frontends,
# so we also inject scoped CSS and attach CSS classes below.
display(HTML(f"""
<style>
/* ------------------------------------------------------------
   ZERO-SUM STYLE FOR IPYWIDGET TOOLBAR
------------------------------------------------------------ */

.zero-sum-root,
.zero-sum-root .widget-box,
.zero-sum-root .widget-hbox,
.zero-sum-root .widget-vbox,
.zero-sum-root .widget-html,
.zero-sum-root .widget-label,
.zero-sum-root .widget-output,
.zero-sum-root .jupyter-widgets {{
    background: {PAPER_BG} !important;
    font-family: "Courier New", monospace !important;
}}

.zero-sum-root {{
    overflow-x: hidden !important;
}}

/* Hide native ipywidgets descriptions.
   We render our own left-side labels via _control_row(...),
   otherwise labels appear twice. */
.zero-sum-root .widget-label {{
    display: none !important;
}}

/* Dropdowns and list selectors */
.zero-sum-root select,
.zero-sum-root input {{
    background: {PAPER_BG} !important;
    color: {TEXT} !important;
    border: 1px solid {GRID} !important;
    border-radius: 0 !important;
    font-family: "Courier New", monospace !important;
    font-size: 12px !important;
    box-shadow: none !important;
}}

.zero-sum-root .zero-sum-ticker-dropdown select,
.zero-sum-root .zero-sum-compare-select select {{
    background: #ffffff !important;
    color: {TEXT} !important;
    border: 1px solid {GRID} !important;
}}

.zero-sum-root .zero-sum-ticker-dropdown select {{
    height: 32px !important;
    padding-left: 8px !important;
}}

.zero-sum-root .zero-sum-compare-select select {{
    min-height: 88px !important;
    padding: 4px 6px !important;
}}

/* Toggle button containers */
.zero-sum-root .widget-toggle-buttons {{
    background: transparent !important;
    gap: 0 !important;
}}

/* Inactive buttons */
.zero-sum-root button,
.zero-sum-root .widget-toggle-button {{
    background: #eee8dd !important;
    color: #262626 !important;
    border: 1px solid #d0c8ba !important;
    border-radius: 0 !important;
    box-shadow: none !important;
    font-family: "Courier New", monospace !important;
    font-size: 11px !important;
    font-weight: 700 !important;
    text-transform: none !important;
    min-height: 30px !important;
    height: 32px !important;
    line-height: 32px !important;
    padding: 0 14px !important;
    margin: 0 !important;
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    vertical-align: middle !important;
}}

.zero-sum-root .widget-toggle-button .widget-label,
.zero-sum-root button .widget-label,
.zero-sum-root .widget-toggle-button span,
.zero-sum-root button span {{
    line-height: 32px !important;
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    height: 100% !important;
    padding: 0 !important;
    margin: 0 !important;
}}

/* Hover */
.zero-sum-root button:hover,
.zero-sum-root .widget-toggle-button:hover {{
    background: #e4ded2 !important;
    border-color: #aaa397 !important;
    color: #1f1f1f !important;
}}

/* Active buttons */
.zero-sum-root .mod-active,
.zero-sum-root .widget-toggle-button.mod-active,
.zero-sum-root button.mod-active,
.zero-sum-root button[aria-pressed="true"] {{
    background: #1f1f1f !important;
    color: #f3eee5 !important;
    border-color: #1f1f1f !important;
}}

/* Remove Bootstrap rounded corners / focus glow */
.zero-sum-root .btn,
.zero-sum-root .btn-default,
.zero-sum-root .btn:focus,
.zero-sum-root button:focus {{
    border-radius: 0 !important;
    outline: none !important;
    box-shadow: none !important;
}}

/* Compact rows */
.zero-sum-root .widget-hbox {{
    align-items: center !important;
    margin-bottom: 6px !important;
}}

/* Output area */
.zero-sum-root .widget-output {{
    border: 0 !important;
}}

/* Plotly modebar: keep reset/home tools, but do not let them dominate the header.
   displayModeBar="hover" handles visibility; these rules make it compact. */
.zero-sum-root .modebar {{
    top: 4px !important;
    right: 8px !important;
    background: transparent !important;
}}

.zero-sum-root .modebar-group {{
    background: rgba(243,238,229,0.80) !important;
}}

.zero-sum-root .modebar-btn svg {{
    opacity: 0.55 !important;
}}

.zero-sum-root .modebar-btn:hover svg {{
    opacity: 1.0 !important;
}}

.zero-sum-chart-output,
.zero-sum-chart-output .widget-output,
.zero-sum-chart-output .output,
.zero-sum-chart-output .output_area,
.zero-sum-chart-output .output_subarea {{
    width: 100% !important;
    max-width: 100% !important;
    min-width: 100% !important;
}}
</style>
"""))


def _active_title():
    if view_widget.value == "Single ticker":
        return ticker_widget.value
    if view_widget.value == "Portfolio":
        return "Portfolio Tracker & Risk Dashboard"
    return "Comparison"


def _render_header():
    active_title = _active_title()
    header_text = _strategy_text(selected_strategy=selected_strategy, risk_level=RISK_LEVEL)

    header_html.value = f"""
    <div style="
        background:{PAPER_BG};
        padding: 18px 24px 10px 24px;
        font-family: 'Courier New', monospace;
        border-bottom: 1px solid {GRID};
    ">
        <div style="
            font-family: Georgia, serif;
            font-size: 34px;
            line-height: 1;
            color: {TEXT};
            margin-bottom: 6px;
        ">{active_title}</div>
        <div style="
            font-size: 13px;
            color: {MUTED};
            white-space: nowrap;
        ">{header_text}</div>
    </div>
    """


def _sync_control_visibility():
    view = view_widget.value

    ticker_row.layout.display = "flex" if view == "Single ticker" else "none"
    compare_row.layout.display = "flex" if view == "Comparison" else "none"

    chart_row.layout.display = "flex" if view == "Single ticker" else "none"
    overlays_row.layout.display = "flex" if view == "Single ticker" else "none"
    studies_row.layout.display = "flex" if view == "Single ticker" else "none"


def _render_chart(_=None):
    _sync_control_visibility()
    _render_header()

    with chart_output:
        clear_output(wait=True)

        try:
            if view_widget.value == "Single ticker":
                fig = build_single_ticker_chart(
                    price_df=raw_df,
                    account_df=df_account_value,
                    action_df=df_actions,
                    ticker=ticker_widget.value,
                    selected_strategy=selected_strategy,
                    risk_level=RISK_LEVEL,
                    period=period_widget.value,
                    chart_type=chart_widget.value,
                    overlay_mode=overlay_widget.value,
                    study_mode=study_widget.value,
                )

            elif view_widget.value == "Portfolio":
                fig = build_portfolio_dashboard_chart(
                    price_df=raw_df,
                    account_df=df_account_value,
                    portfolio_input=globals().get("portfolio_input", None),
                    period=period_widget.value,
                )

            else:
                selected_compare_tickers = list(compare_tickers_widget.value)
                if len(selected_compare_tickers) == 0:
                    selected_compare_tickers = available_tickers

                fig = build_normalized_comparison_chart(
                    price_df=raw_df,
                    account_df=df_account_value,
                    selected_tickers=selected_compare_tickers,
                    period=period_widget.value,
                )

            fig.show(config=PLOTLY_CONFIG)

        except Exception as e:
            active_view = view_widget.value
            active_period = period_widget.value
            active_ticker = ticker_widget.value if "ticker_widget" in globals() else None
            active_compare = list(compare_tickers_widget.value) if "compare_tickers_widget" in globals() else []

            print("=" * 100)
            print("CHART DEBUG ERROR")
            print("=" * 100)
            print("View:", active_view)
            print("Period:", active_period)
            print("Ticker:", active_ticker)
            print("Compare tickers:", active_compare)
            print("raw_df shape:", getattr(raw_df, "shape", None))
            print("raw_df columns:", list(raw_df.columns) if hasattr(raw_df, "columns") else None)
            print("df_account_value shape:", getattr(df_account_value, "shape", None))
            print("df_account_value columns:", list(df_account_value.columns) if hasattr(df_account_value, "columns") else None)
            print("portfolio_input:", globals().get("portfolio_input", None))
            print("Error type:", type(e).__name__)
            print("Error message:", str(e))
            print("Traceback:")
            print(traceback.format_exc())

            display(HTML(f"""
            <div style="
                background:#fff3cd;
                border:1px solid #e0b84f;
                padding:12px 14px;
                color:#262626;
                font-family:'Courier New', monospace;
                white-space:pre-wrap;
            ">
                <b>Chart error:</b> {type(e).__name__}: {e}<br>
                <span style="color:#7a766e;">Se debug-loggen lige over denne boks for view, data shapes, kolonner og traceback.</span>
            </div>
            """))


def _control_row(label, widget):
    return widgets.HBox(
        [
            widgets.HTML(
                f"<div style='width:90px;color:{MUTED};font-family:Courier New,monospace;font-size:11px;text-transform:uppercase;letter-spacing:0.08em;font-weight:700;'>{label}</div>"
            ),
            widget,
        ],
        layout=widgets.Layout(align_items="center", margin="2px 0"),
    )


view_row = _control_row("VIEW", view_widget)
ticker_row = _control_row("TICKER", ticker_widget)
compare_row = _control_row("COMPARE", compare_tickers_widget)
period_row = _control_row("PERIOD", period_widget)
chart_row = _control_row("CHART", chart_widget)
overlays_row = _control_row("OVERLAYS", overlay_widget)
studies_row = _control_row("STUDIES", study_widget)

controls = widgets.VBox(
    [
        view_row,
        ticker_row,
        compare_row,
        period_row,
        chart_row,
        overlays_row,
        studies_row,
    ],
    layout=widgets.Layout(
        padding="14px 24px 10px 24px",
        border="0",
    ),
)

controls.add_class("zero-sum-controls")
chart_output.add_class("zero-sum-chart-output")

container = widgets.VBox(
    [header_html, controls, chart_output],
    layout=widgets.Layout(
        width="100%",
        border="0",
        padding="0",
        margin="0",
    ),
)

container.add_class("zero-sum-root")

# Re-render on changes. Widgets stay fixed; only chart_output is refreshed.
for widget in [
    view_widget,
    ticker_widget,
    compare_tickers_widget,
    period_widget,
    chart_widget,
    overlay_widget,
    study_widget,
]:
    widget.observe(_render_chart, names="value")

# Initial view
_sync_control_visibility()
_render_header()
display(container)

# Give the frontend a moment to mount the widget container before rendering Plotly.
# This prevents the first render from being empty or width-broken in Jupyter/VS Code.
import threading
import time

def _initial_render_after_mount():
    time.sleep(0.25)
    _render_chart()

threading.Thread(target=_initial_render_after_mount).start()


In [ ]:
def plot_action_distribution(action_df):
    counts = action_df["action_name"].value_counts().reset_index()
    counts.columns = ["action", "count"]
    fig = go.Figure(go.Bar(x=counts["action"], y=counts["count"], marker_color="#42a5f5"))
    fig.update_layout(title="PPO high-level action distribution", template="plotly_dark", height=420, paper_bgcolor="#0f172a", plot_bgcolor="#0f172a", font=dict(color="#e5e7eb"), margin=dict(l=40, r=40, t=70, b=40))
    return fig

plot_action_distribution(df_actions).show()

In [ ]:
def plot_strategy_timeline(action_df):
    df = action_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["strategy_code"] = pd.Categorical(df["strategy_id"], categories=list(STRATEGIES.keys())).codes
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["date"], y=df["strategy_code"], mode="lines+markers", name="Strategy", line=dict(width=2, color="#f0b90b"), marker=dict(size=5), text=df["strategy_id"], hovertemplate="%{x}<br>Strategy: %{text}<extra></extra>"))
    fig.update_yaxes(tickmode="array", tickvals=list(range(len(STRATEGIES))), ticktext=list(STRATEGIES.keys()))
    fig.update_layout(title="Strategy state over backtest", template="plotly_dark", height=360, paper_bgcolor="#0f172a", plot_bgcolor="#0f172a", font=dict(color="#e5e7eb"), margin=dict(l=40, r=40, t=70, b=40))
    return fig

plot_strategy_timeline(df_actions).show()

In [ ]:
os.makedirs("outputs", exist_ok=True)
df_account_value.to_csv("outputs/ppo_account_value.csv", index=False)
df_actions.to_csv("outputs/ppo_strategy_actions.csv", index=False)
print("Saved outputs")